In [11]:
import pandas as pd
import emoji
import nltk
import torch
import sys
import shutil
import urllib
import tarfile
from pathlib import Path
import numpy as np
from typing import Iterable
from tqdm import tqdm
from collections import Counter

import re

## hashtags segmenter
import wordsegment as ws

In [12]:
# choose right device
device = torch.device('cpu')
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device("mps")
print('Torch Device: %s' % device)

Torch Device: cpu


# Data Visualization

In [13]:
# Task 2 -remove emojis
def contains_emoji(text):
    return any(emoji.is_emoji(char) for char in text)

def emoji2description(text):
    return emoji.replace_emoji(text, replace=lambda chars, data_dict: ' '.join(data_dict['en'].split('_')).strip(':')+' ')
  
def encode_emojis(data,column='tweet'):
    data[column] = data[column].apply(emoji2description)
    return data


# Task 2 - replace URL with a token
def replace_urls_with_token(text,token):
    # Regular expression pattern for matching URLs
    url_pattern = r'https?://(?:www\.)?[\w-]+\.[\w/._-]+'
    
    # Replace URLs with the token
    return re.sub(url_pattern, token, text)

def encode_URL(data, column='tweet'):
    data[column] = data[column].apply(replace_urls_with_token)
    return data



In [14]:
def majority_vote(lst):
    count = Counter(lst)
    return count.most_common(1)[0][0]

def yes_probability(lst):
    # Count the number of YES and NO votes
    yes_count = lst.count("YES")
    no_count = lst.count("NO")
    
    # Calculate the probability of YES
    total = yes_count + no_count
    if total == 0:
        return None  # To handle cases where there are no "YES" or "NO" votes in the list
    else:
        return yes_count / total

def generate_labels_task1(data):
    data = data.drop(['annotators','number_annotators','gender_annotators','age_annotators','labels_task2','labels_task3'],axis=1)
    data['hard_label_task1'] = data['labels_task1'].apply(majority_vote)
    data['soft_label_task1'] = data['labels_task1'].apply(yes_probability)
    return data

def load_and_preprocess_dataset(file_path):
    data = pd.read_json(file_path).T
    data = generate_labels_task1(data)
    columns_to_keep = ['id_EXIST','lang','tweet', 'hard_label_task1','soft_label_task1']
    data = data[columns_to_keep]
    data = filter_Df(data)
    data = encode_emojis(data)
    return data

def filter_Df(df):
    new_df = df[df['lang'].astype('str').str.contains('en')]
    return new_df


In [15]:
training_data = load_and_preprocess_dataset("data/training.json")
validation_data = load_and_preprocess_dataset("data/validation.json")
test_data = load_and_preprocess_dataset("data/test.json")

In [16]:
print(training_data.shape)
print(validation_data.shape)
print(test_data.shape)

print(training_data.columns)


(3260, 5)
(177, 5)
(312, 5)
Index(['id_EXIST', 'lang', 'tweet', 'hard_label_task1', 'soft_label_task1'], dtype='object')


In [17]:
display(training_data.head())

,id_EXIST,lang,tweet,hard_label_task1,soft_label_task1
200001,200001,en,FFS! How about laying the blame on the bastard...,YES,0.500000
200002,200002,en,Writing a uni essay in my local pub with a cof...,YES,0.833333
200003,200003,en,@UniversalORL it is 2021 not 1921. I dont appr...,YES,0.666667
200004,200004,en,@GMB this is unacceptable. Use her title as yo...,YES,0.500000
200005,200005,en,‘Making yourself a harder target’ basically bo...,YES,0.500000


In [18]:
display(test_data.head())

,id_EXIST,lang,tweet,hard_label_task1,soft_label_task1
400178,400178,en,1st day at the pool on a beautiful Sunday in N...,NO,0.000000
400179,400179,en,“I like your outfit too except when i dress up...,YES,0.833333
400180,400180,en,@KNasFanFic pleading face sparkling heart sam...,NO,0.000000
400181,400181,en,@themaxburns @GOP Fuck that cunt. Tried to vot...,YES,0.833333
400182,400182,en,@ultshunnie u gotta say some shit like “i’ll f...,YES,1.000000


In [19]:
print(training_data.isna().sum().sum(), test_data.isna().sum().sum(), validation_data.isna().sum().sum())

0 0 0


# Data Cleaning

## Emojis
[source1](https://towardsdatascience.com/emojis-aid-social-media-sentiment-analysis-stop-cleaning-them-out-bb32a1e5fc8e)
[source2](https://medium.com/@HeCanThink/emoji-lets-deal-with-in-python-dcc88b1f8ab1)
[source3](https://kt.ijs.si/data/Emoji_sentiment_ranking/)

In [20]:
#TODO sentiment analysis on emojis
def add_sentiment_analysis_from_emojis(text):
    return text

def unicode_escape(chars, data_dict):
    return hex(ord(chars.encode('unicode-escape').decode()))

emoji_char = "😊"

# Get Unicode representation of the emoji
emoji_unicode = emoji.replace_emoji(emoji_char, replace=unicode_escape)
print(emoji_unicode)

TypeError: ord() expected a character, but string of length 10 found

## Mention

In [21]:
import spacy
nlp = spacy.load("en_core_web_sm")

In [22]:
mention_tweets =  training_data["tweet"]

In [23]:
def get_mention(sentences):
    MENTION_REGEX = r"@\w+"
    tweets = []
    for sen in sentences: # sen is a single sentence, snetences are all the tweets
        mention = re.findall(MENTION_REGEX, sen)
        if mention:
            new_sen = sen
            for m in mention:
                doc = nlp(m)
                sub = [c.label_ for c in doc.ents]
                if sub:
                    new_sen = new_sen.replace(m, sub[0])
                else:
                    new_sen = new_sen.replace(m, [c.pos_ for c in doc][0])
            tweets.append(new_sen)
        else:
            tweets.append(sen)
    return np.array(tweets)


# exemple of the function
tweets = get_mention(mention_tweets.to_numpy()[25:31])
print(f"Mention found: {tweets.size}")

Mention found: 6


In [24]:
print(mention_tweets.to_numpy()[25:31]) 

["@HKesvani It's especially weird because the alt-right on YouTube is older than Breadtube, which was a reaction to RW gamergate content."
 "@Women_Truths Well, they're kind of shooting themselves in the foot by injecting political drivel into entertainment. By pushing stress into escapism, you impassion people by stripping them of their release...thus spawning movements like GamerGate, ComicsGate, etc... People pushing back. (c)"
 "@Argumentaleon @BlazingTamaki @ColdCriti I don't even know who tf they are. Only know they were part of the whole gamergate shitstorm back in 2015."
 "as a person who lived through gamergate, these new generation incels bitching about #HorizonForbiddenWest Aloy's peachfuzz is not what I or my hairy russian futanari with shitting dick nipples hentai addicted brothers were fighting for.Give Aloy a beard and a huge cock."
 "Anita as an asset? Damn I should've considered that. Her vs gamergate was one big spectacle https://t.co/x2k5KDnUt2"
 "@punishedmother I W

In [25]:
print(tweets)

["NOUN It's especially weird because the alt-right on YouTube is older than Breadtube, which was a reaction to RW gamergate content."
 "ORG Well, they're kind of shooting themselves in the foot by injecting political drivel into entertainment. By pushing stress into escapism, you impassion people by stripping them of their release...thus spawning movements like GamerGate, ComicsGate, etc... People pushing back. (c)"
 "ORG PROPN ORG I don't even know who tf they are. Only know they were part of the whole gamergate shitstorm back in 2015."
 "as a person who lived through gamergate, these new generation incels bitching about #HorizonForbiddenWest Aloy's peachfuzz is not what I or my hairy russian futanari with shitting dick nipples hentai addicted brothers were fighting for.Give Aloy a beard and a huge cock."
 "Anita as an asset? Damn I should've considered that. Her vs gamergate was one big spectacle https://t.co/x2k5KDnUt2"
 "NOUN I WANT TO GET OFF 2014I WANT TO GET OFF GAMERGATE'S WILD

To all the training dataset

In [26]:
tweets = get_mention(mention_tweets.to_numpy()) 

## Hastags

Thanks to [hashformers](https://github.com/ruanchaves/hashformers) of Ruan Chaves Rodrigues, Marcelo Akira Inuzuka, Juliana Resplande Sant'Anna Gomes, Acquila Santos Rocha, Iacer Calixto, Hugo Alexandre Dantas do Nascimento.

ijijk

In [ ]:
hastag_tweets = training_data["tweet"]

In [ ]:
def get_hashtags(sentences):
    HASHTAG_REGEX = r"#\w+"
    htags = []
    for sen in sentences:
        htags += re.findall(HASHTAG_REGEX, sen)
    return np.array(htags)

def segment(sentences):
    segmented_sen = []
    for sen in sentences:
        segmented_sen += [np.array(ws.segment(sen))]
    return np.array(segmented_sen, dtype=object)


In [ ]:
htags_raw = get_hashtags(hastag_tweets.to_numpy())
print(f"Hashtags found: {htags_raw.size}")
print(f"Hashtag before: {htags_raw}")

Hashtags found: 1293
Hashtag before: ['#EverydaySexism' '#everydaysexism' '#everydaysexism' ... '#goldengirls'
 '#StandWithUkriane' '#GoldenGirls']


In [ ]:
htags = segment(htags_raw)
print(f"Hashtag after: {htags}")

Hashtag after: [array(['everyday', 'sexism'], dtype='<U8')
 array(['everyday', 'sexism'], dtype='<U8')
 array(['everyday', 'sexism'], dtype='<U8') ...
 array(['golden', 'girls'], dtype='<U6')
 array(['stand', 'with', 'uk', 'riane'], dtype='<U5')
 array(['golden', 'girls'], dtype='<U6')]
